# Inference Configuration
Specify
- path

In [1]:
path = "../../output/protenn2/v5"


In [2]:
import json
import os.path
import pickle

from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

from src.protenn2.utils import get_train_val_test_paths, get_device

# get file paths

log_path = os.path.join(path, "log")

label_encoder_path = os.path.join(path, "label_encoder.pkl")
model_path = os.path.join(path, "best_model.pt")
with open(os.path.join(path, "params.json"), "r") as f:
    params = json.load(f)
if "input_folder" not in params:
    raise ValueError("input_folder must be specified")
dataset_path = os.path.join("../../", params["input_folder"])
train_path, val_path, test_path = get_train_val_test_paths(dataset_path)


In [3]:
from src.protenn2.utils import calculate_max_protein_length
from src.protenn2.dataset import CathPredPerResidueDataset, create_protein_collate_fn
from src.protenn2.analysis.cath_hierarchy_mapper import CATHHierarchyMapper

# Initialize objects

device = get_device()
with open(label_encoder_path, "rb") as f:
    label_encoder: LabelEncoder = pickle.load(f)
num_classes = len(label_encoder.classes_)
max_protein_length = calculate_max_protein_length(dataset_path)

test_dataset = CathPredPerResidueDataset(test_path, label_encoder=label_encoder,
                                         embedding_dir="../../data/embeddings/protein_embeddings_new")

collate_fn = create_protein_collate_fn(max_protein_length, test_dataset.padding_encoded_id)

test_dataloader = DataLoader(test_dataset, collate_fn=collate_fn)

mapper = CATHHierarchyMapper(label_encoder=label_encoder)

Using MPS (Apple Silicon GPU).
Max protein length: 599
Dataset initialized with 1317 unique proteins.


In [4]:
import numpy as np

test_dataset2 = CathPredPerResidueDataset(test_path, label_encoder=label_encoder,
                                          embedding_dir="../../data/embeddings/protein_embeddings_new")
y_true_labels_list = []
for _, y_true_labels, _ in test_dataset2:
    y_true_labels_list.append(y_true_labels)

y_true_flat = np.concatenate(y_true_labels_list)
classes, counts = np.unique(y_true_flat, return_counts=True)
global_distribution = np.zeros(test_dataset.num_classes)
global_distribution[classes] = counts / counts.sum()

Dataset initialized with 1317 unique proteins.


In [5]:
# y_true_labels_list_c = [mapper.map_labels(labels, "C") for labels in y_true_labels_list]
# y_true_flat = np.concatenate(y_true_labels_list_c)
# classes, counts = np.unique(y_true_flat, return_counts=True)
# global_distribution = np.zeros(mapper.get_class_count("C"))
# global_distribution[classes] = counts
# global_distribution

In [6]:
dummy_stratified_classifier_per_protein_kwargs = {"distribution": global_distribution}
dummy_majority_classifier_per_protein_kwargs = {"majority_label": 1}

In [7]:
from src.protenn2.analysis.inference import run_inference_dummy, dummy_stratified_classifier_per_protein

y_true_labels_list, y_pred_confidences_list, protein_chain_id_list = run_inference_dummy(
    dummy_classifier=dummy_stratified_classifier_per_protein,
    dataloader=test_dataloader,
    padding_encoded_id=test_dataset.padding_encoded_id,
    num_classes=mapper.get_class_count(target_hierarchy="H"),
    dummy_classifier_kwargs=dummy_stratified_classifier_per_protein_kwargs)

Running dummy inference on 1317 proteins...


Dummy Inference Progress: 100%|██████████| 1317/1317 [00:01<00:00, 761.84it/s]

Dummy inference complete. Processed 1317 proteins.


# Analysis Configuration

In [14]:


bootstrap_samples = 1000
post_process_kwargs = None
post_process_func = None
# metrics_to_compute = ("accuracy", "f1_score", "jaccard_score", "recall_score", "precision_score",
#                       "segment_overlap_score")
metrics_to_compute = ("accuracy", "jaccard_score",
                      "segment_overlap_score")

In [15]:
from src.protenn2.analysis.metrics import calculate_metrics_for_cath_levels

all_results = calculate_metrics_for_cath_levels(y_true_labels_list=y_true_labels_list,
                                                y_pred_confidences_list=y_pred_confidences_list, mapper=mapper,
                                                bootstrap_samples=bootstrap_samples,
                                                post_process_func=None,
                                                post_process_kwargs=None,
                                                metrics_to_compute=metrics_to_compute,
                                                levels=("C", "A", "T", "H"),
                                                name="baseline_stratified")

---- Computing Metrics for hierarchy: C
('accuracy', 'jaccard_score', 'segment_overlap_score')
----- Accuracy Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:02<00:00, 345.34it/s]


{'mean': np.float64(0.2879258455708221), 'ci_lower': np.float64(0.28299163836986285), 'ci_upper': np.float64(0.29240121193282276), 'alpha': 0.05}
----- Jaccard Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:14<00:00, 70.55it/s]


{'mean': np.float64(0.15821616818278106), 'ci_lower': np.float64(0.1508438475170919), 'ci_upper': np.float64(0.1658569006065693), 'alpha': 0.05}
----- Segment Overlap Score Metrics -----


...

{'mean': np.float64(0.11033847375234725), 'ci_lower': np.float64(0.10362388457520949), 'ci_upper': np.float64(0.11648379493156698), 'alpha': 0.05}
----- Jaccard Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:40<00:00, 24.40it/s]


{'mean': np.float64(0.00066979246419944), 'ci_lower': np.float64(0.0005717490227164822), 'ci_upper': np.float64(0.0007772693728665545), 'alpha': 0.05}
----- Segment Overlap Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [01:43<00:00,  9.65it/s]

{'mean': np.float64(0.0009413120694108793), 'ci_lower': np.float64(0.0008251276245953621), 'ci_upper': np.float64(0.0010565761340606387), 'alpha': 0.05}


# All results

In [16]:
all_results

{'baseline_stratified_C': {'accuracy': {'mean': np.float64(0.2879258455708221),
   'ci_lower': np.float64(0.28299163836986285),
   'ci_upper': np.float64(0.29240121193282276),
   'alpha': 0.05},
  'jaccard_score': {'mean': np.float64(0.15821616818278106),
   'ci_lower': np.float64(0.1508438475170919),
   'ci_upper': np.float64(0.1658569006065693),
   'alpha': 0.05},
  'segment_overlap_score': {'mean': np.float64(0.03139716197899321),
   'ci_lower': np.float64(0.030371250969068526),
   'ci_upper': np.float64(0.03248018859159994),
   'alpha': 0.05}},
 'baseline_stratified_A': {'accuracy': {'mean': np.float64(0.15542782607021857),
   'ci_lower': np.float64(0.1498048262113331),
   'ci_upper': np.float64(0.16063667901475076),
   'alpha': 0.05},
  'jaccard_score': {'mean': np.float64(0.03609921089018046),
   'ci_lower': np.float64(0.03387514138976865),
   'ci_upper': np.float64(0.03836186603341956),
   'alpha': 0.05},
  'segment_overlap_score': {'mean': np.float64(0.012586217849723962),
   '

In [17]:
with open(os.path.join(path, "baseline_test_metrics_strat.json"), "w") as f:
    json.dump(all_results, f)